# Continued Fine-Tuning — NLLB + LoRA, éwé (low-resource MT)

**Projet GR02 · extension recherche.** Ce notebook **n'entraîne pas un modèle depuis
zéro** : il **reprend** un adaptateur LoRA déjà entraîné et **poursuit** son entraînement
(*continued fine-tuning*) sur le corpus **augmenté par back-translation**. La sortie va
dans un dossier **séparé** (`models/nllb-ewe-lora-cft`) ; aucun modèle existant n'est écrasé.

Il met en œuvre trois ingrédients classiques des travaux de **traduction automatique pour
langues peu dotées** :

1. **Continued fine-tuning / curriculum** — on part du modèle entraîné sur les données
   propres, puis on poursuit sur les données augmentées (deux étapes plutôt qu'un seul jet).
2. **Augmentation par back-translation** — éwé humain en cible, source synthétique générée
   par le meilleur modèle courant (option d'itération *round 2*).
3. **Évaluation honnête sur FLORES-200** — benchmark externe, six directions, BLEU + chrF,
   et **étude comparative** des régimes d'entraînement (propre / back-translation / continué).

## 0. Principe

Trois modèles sont comparés sur le **même** jeu de test :

| Régime | Données | Départ |
|---|---|---|
| LoRA propre | corpus propre | NLLB de base |
| Back-translation | corpus augmenté | NLLB de base (from scratch) |
| **Continué (ce notebook)** | corpus augmenté | **modèle back-translation** (le meilleur) |

Le *continued fine-tuning* utilise un **taux d'apprentissage réduit** (on affine, on ne
réapprend pas) et **moins d'époques**, puisqu'on part d'un modèle déjà compétent.

In [1]:
from pathlib import Path
import os, sys, time

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "nllb"
TRAIN_CSV = DATA_DIR / "nllb_train_aug.csv"
VALID_CSV = DATA_DIR / "nllb_valid.csv"
TEST_CSV = DATA_DIR / "nllb_test.csv"

MODEL_NAME = "facebook/nllb-200-distilled-600M"
START_ADAPTER = PROJECT_ROOT / "models" / "nllb-ewe-lora-bt"
OUTPUT_DIR = PROJECT_ROOT / "models" / "nllb-ewe-lora-cft"
COMPARE_DIRS = {
    "LoRA propre": PROJECT_ROOT / "models" / "nllb-ewe-lora" / "evaluation_report.csv",
    "Back-translation": PROJECT_ROOT / "models" / "nllb-ewe-lora-bt" / "evaluation_report.csv",
}

MODE = "full"
REQUIRE_CUDA = True
IS_SMOKE = MODE == "smoke"
MAX_TRAIN_SAMPLES = 512 if IS_SMOKE else 800000
MAX_VALID_SAMPLES = 128 if IS_SMOKE else 6000
MAX_EVAL_PER_DIRECTION = 20 if IS_SMOKE else 200
EPOCHS = 0.05 if IS_SMOKE else 1.0
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
LEARNING_RATE = 1e-4
FLORES_LANGS = {"eng": "eng_Latn", "fra": "fra_Latn", "ewe": "ewe_Latn"}

print("Départ (adaptateur):", START_ADAPTER.name)
print("Sortie (continué)  :", OUTPUT_DIR.name)
print("Données            :", TRAIN_CSV.name, "| LR continué:", LEARNING_RATE, "| époques:", EPOCHS)

Départ (adaptateur): nllb-ewe-lora-bt
Sortie (continué)  : nllb-ewe-lora-cft
Données            : nllb_train_aug.csv | LR continué: 0.0001 | époques: 1.0


## 1. Diagnostic GPU

In [2]:
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA indisponible. Installez PyTorch CUDA avant le continued fine-tuning.")

torch: 2.11.0+cu130 | cuda: True
device: NVIDIA GeForce RTX 5080 Laptop GPU | bf16: True


## 2. Données augmentées

On reprend le corpus **augmenté par back-translation** (`nllb_train_aug.csv`, produit par
le notebook 02). Les ensembles `valid` et `test` restent identiques à tous les autres
modèles : la comparaison est donc honnête, sans fuite.

> *Itération facultative (round 2)* : pour se rapprocher davantage de l'état de l'art, on
> peut régénérer le synthétique avec le **meilleur modèle courant** avant de poursuivre.
> Cette étape est lourde (GPU) et désactivée par défaut.

### Régénération itérative — back-translation round 2 (optionnel)

Le modèle back-translation est désormais meilleur que celui ayant servi au round 1. On
peut donc **régénérer des sources synthétiques de meilleure qualité**, sur un **échantillon
plus large** du pool éwé pristin, puis **fusionner** avec le corpus augmenté existant.

Deux garde-fous indispensables :
- **Pas de doublons** : déduplication sur `(src_lang, tgt_lang, src_text, tgt_text)` après fusion.
- **Cible toujours humaine** : seule la source (en/fr) est synthétique, l'éwé reste humain.

Coûteux (génération GPU) → désactivé par défaut (`RUN_ROUND2 = False`). Activé, il écrit
`nllb_train_aug2.csv` (plus volumineux que le round 1) et l'entraînement plus bas l'utilise
automatiquement.

In [3]:
import pandas as pd

RUN_ROUND2 = True
ROUND2_SAMPLE = 100000
ROUND2_OUT = DATA_DIR / "nllb_train_aug2.csv"
BEST_ADAPTER = PROJECT_ROOT / "models" / "nllb-ewe-lora-bt"

if RUN_ROUND2:
    import hashlib
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    from peft import PeftModel

    NLLB = {"fra": "fra_Latn", "eng": "eng_Latn", "ewe": "ewe_Latn"}
    EWE_CHARS = set("ɖƒŋɔɛɣʋ̃")
    R2_BATCH, R2_BEAMS = 48, 4
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    g_tok = AutoTokenizer.from_pretrained(str(BEST_ADAPTER))
    g_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    g_model = PeftModel.from_pretrained(g_model, str(BEST_ADAPTER)).to(dev).eval()
    try:
        g_model.generation_config.max_length = None
    except Exception:
        pass
    g_bf16 = dev == "cuda" and torch.cuda.is_bf16_supported()

    @torch.no_grad()
    def r2_translate(texts, src, tgt):
        g_tok.src_lang = NLLB[src]
        bos = g_tok.convert_tokens_to_ids(NLLB[tgt])
        out = []
        ctx = torch.autocast("cuda", dtype=torch.bfloat16) if g_bf16 else torch.autocast("cpu", enabled=False)
        for i in range(0, len(texts), R2_BATCH):
            batch = [str(x) for x in texts[i:i + R2_BATCH]]
            enc = g_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
            with ctx:
                gen = g_model.generate(**enc, forced_bos_token_id=bos, max_new_tokens=128, num_beams=R2_BEAMS)
            out.extend(g_tok.batch_decode(gen, skip_special_tokens=True))
            if (i // R2_BATCH) % 50 == 0:
                print(f"  {src}->{tgt}: {min(i + R2_BATCH, len(texts))}/{len(texts)}", flush=True)
        return out

    clean = pd.read_csv(DATA_DIR / "nllb_train.csv")
    pool = clean[(clean.src_lang == "ewe") & (clean.tgt_lang == "fra")][["src_text", "tgt_text", "quality_score"]].copy()
    pool.columns = ["ewe", "fr", "cos"]
    pool = pool.dropna().drop_duplicates(subset=["ewe", "fr"])
    pool = pool[(pool["cos"].astype(float) >= 0.65) & pool["ewe"].map(lambda s: any(c in EWE_CHARS for c in str(s)))]
    pool = pool.sample(n=min(ROUND2_SAMPLE, len(pool)), random_state=7).reset_index(drop=True)
    print("Pool round 2 (paires éwé pristines) :", len(pool))

    def r2_sid(*parts):
        return hashlib.sha1("␟".join(str(p) for p in parts).encode("utf-8")).hexdigest()

    new_rows = []
    def r2_emit(sl, tl, srcs, tgts, tag):
        for s, e in zip(srcs, tgts):
            s, e = str(s).strip(), str(e).strip()
            if not s or not e:
                continue
            new_rows.append({"id": r2_sid("train", sl, tl, s, e), "split": "train",
                             "src_lang": sl, "tgt_lang": tl,
                             "src_lang_nllb": NLLB[sl], "tgt_lang_nllb": NLLB[tl],
                             "src_text": s, "tgt_text": e, "source": tag, "domain": "synthetic",
                             "quality_score": 1.0, "is_religious": False, "license": ""})

    en_synth = r2_translate(pool["fr"].tolist(), "fra", "eng")
    r2_emit("eng", "ewe", en_synth, pool["ewe"].tolist(), "synthetic:bt2:fr2en")
    r2_emit("ewe", "eng", pool["ewe"].tolist(), en_synth, "synthetic:bt2:pivot")
    fr_synth = r2_translate(pool["ewe"].tolist(), "ewe", "fra")
    r2_emit("fra", "ewe", fr_synth, pool["ewe"].tolist(), "synthetic:bt2:ewe2fr")
    r2_new = pd.DataFrame(new_rows)

    base_aug = pd.read_csv(TRAIN_CSV) if TRAIN_CSV.exists() else pd.read_csv(DATA_DIR / "nllb_train.csv")
    combined = pd.concat([base_aug, r2_new], ignore_index=True)
    before = len(combined)
    combined = combined.drop_duplicates(subset=["src_lang", "tgt_lang", "src_text", "tgt_text"]).reset_index(drop=True)
    combined.to_csv(ROUND2_OUT, index=False, encoding="utf-8")
    print(f"Round 2 : {len(r2_new)} paires générées | fusion {before} -> {len(combined)} après déduplication")
    print(f"Gain net vs corpus augmenté de départ : +{len(combined) - len(base_aug)} paires")
    print(combined.groupby(["src_lang", "tgt_lang"]).size().to_string())
    TRAIN_CSV = ROUND2_OUT

    del g_model, g_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Round 2 désactivé — on utilise", TRAIN_CSV.name)

D:\python\traducteur-fr-en-ewe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 51067.34it/s]


Pool round 2 (paires éwé pristines) : 100000
  fra->eng: 48/100000
  fra->eng: 2448/100000
  fra->eng: 4848/100000
  fra->eng: 7248/100000
  fra->eng: 9648/100000
  fra->eng: 12048/100000
  fra->eng: 14448/100000
  fra->eng: 16848/100000
  fra->eng: 19248/100000
  fra->eng: 21648/100000
  fra->eng: 24048/100000
  fra->eng: 26448/100000
  fra->eng: 28848/100000
  fra->eng: 31248/100000
  fra->eng: 33648/100000
  fra->eng: 36048/100000
  fra->eng: 38448/100000
  fra->eng: 40848/100000
  fra->eng: 43248/100000
  fra->eng: 45648/100000
  fra->eng: 48048/100000
  fra->eng: 50448/100000
  fra->eng: 52848/100000
  fra->eng: 55248/100000
  fra->eng: 57648/100000
  fra->eng: 60048/100000
  fra->eng: 62448/100000
  fra->eng: 64848/100000
  fra->eng: 67248/100000
  fra->eng: 69648/100000
  fra->eng: 72048/100000
  fra->eng: 74448/100000
  fra->eng: 76848/100000
  fra->eng: 79248/100000
  fra->eng: 81648/100000
  fra->eng: 84048/100000
  fra->eng: 86448/100000
  fra->eng: 88848/100000
  fra->eng: 

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_28208\2375771249.py:73: DtypeWarning: Columns (0: domain) have mixed types. Specify dtype option on import or set low_memory=False.
  base_aug = pd.read_csv(TRAIN_CSV) if TRAIN_CSV.exists() else pd.read_csv(DATA_DIR / "nllb_train.csv")


Round 2 : 300000 paires générées | fusion 1144446 -> 991522 après déduplication
Gain net vs corpus augmenté de départ : +147076 paires
src_lang  tgt_lang
eng       ewe         227087
          fra         115223
ewe       eng         227087
          fra         115184
fra       eng         115223
          ewe         191718


In [4]:
import pandas as pd

if not TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"{TRAIN_CSV} introuvable. Génère d'abord les données augmentées "
        "(notebook 02, section 2)."
    )

def read_training_csv(path, max_samples=None):
    df = pd.read_csv(path)
    df["src_text"] = df["src_text"].fillna("").astype(str).str.strip()
    df["tgt_text"] = df["tgt_text"].fillna("").astype(str).str.strip()
    df = df[(df["src_text"].str.len() > 0) & (df["tgt_text"].str.len() > 0)]
    if max_samples:
        df = df.sample(n=min(max_samples, len(df)), random_state=42)
    return df.reset_index(drop=True)

train_df = read_training_csv(TRAIN_CSV, MAX_TRAIN_SAMPLES)
valid_df = read_training_csv(VALID_CSV, MAX_VALID_SAMPLES)
print("train:", len(train_df), "| valid:", len(valid_df))
display(train_df["src_lang"].str.cat(train_df["tgt_lang"], sep="->").value_counts())

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_28208\665925112.py:10: DtypeWarning: Columns (0: domain) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


train: 800000 | valid: 6000


src_lang
eng->ewe    183049
ewe->eng    182956
fra->ewe    155032
ewe->fra     93223
eng->fra     92946
fra->eng     92794
Name: count, dtype: int64

## 3. Reprise du modèle (continued fine-tuning)

Point clé : on charge le modèle de base **puis l'adaptateur LoRA existant en mode
entraînable** (`is_trainable=True`). On **continue** donc à optimiser les mêmes matrices
LoRA, au lieu d'en initialiser de nouvelles. `print_trainable_parameters()` confirme que
seules les matrices LoRA sont entraînées (le reste de NLLB reste gelé).

In [5]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(str(START_ADAPTER))
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
base_model.generation_config.max_length = None
if MODE == "full":
    base_model.gradient_checkpointing_enable()
    base_model.config.use_cache = False

model = PeftModel.from_pretrained(base_model, str(START_ADAPTER), is_trainable=True)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 20381.18it/s]


trainable params: 17,301,504 || all params: 632,375,296 || trainable%: 2.7360


## 4. Tokenisation

Identique aux autres notebooks : chaque ligne porte `src_lang_nllb` et `tgt_lang_nllb`,
injectés dans le tokenizer SentencePiece de NLLB.

In [6]:
def preprocess(example):
    tokenizer.src_lang = example["src_lang_nllb"]
    tokenizer.tgt_lang = example["tgt_lang_nllb"]
    model_inputs = tokenizer(example["src_text"], max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(text_target=example["tgt_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(
    preprocess, remove_columns=train_df.columns.tolist(), desc="Tokenisation train")
valid_ds = Dataset.from_pandas(valid_df, preserve_index=False).map(
    preprocess, remove_columns=valid_df.columns.tolist(), desc="Tokenisation valid")

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print(train_ds)

Tokenisation valid: 100%|██████████| 6000/6000 [00:01<00:00, 4957.67 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 800000
})


## 5. Entraînement continué (taux d'apprentissage réduit)

On poursuit avec `LEARNING_RATE = 1e-4` (la moitié de l'entraînement initial) et **1 époque** :
on affine un modèle déjà compétent, on ne le réapprend pas. La sortie va dans
`models/nllb-ewe-lora-cft`.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
eval_steps = 25 if IS_SMOKE else 1000
save_steps = 50 if IS_SMOKE else 1000
logging_steps = 5 if IS_SMOKE else 50

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.03,
    fp16=use_fp16,
    bf16=use_bf16,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=eval_steps,
    save_steps=save_steps,
    logging_steps=logging_steps,
    save_total_limit=2,
    predict_with_generate=False,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=collator,
)

start = time.time()
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Temps (min):", round((time.time() - start) / 60, 2), "| modèle continué dans", OUTPUT_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
1000,2.467791,1.079791
2000,2.399028,1.077876
3000,2.399641,1.075059
4000,2.413143,1.071974
5000,2.421642,1.070655
6000,2.410834,1.067694
7000,2.396677,1.066818
8000,2.365437,1.064023


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 6. Courbes d'entraînement

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history.to_csv(OUTPUT_DIR / "training_log.csv", index=False)
display(history.tail(12))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "loss" in history:
    history.dropna(subset=["loss"]).plot(x="step", y="loss", ax=axes[0], title="Train loss (continué)")
if "eval_loss" in history:
    history.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=axes[1], title="Validation loss (continué)")
plt.tight_layout(); plt.show()

## 7. Évaluation FLORES-200 : NLLB de base vs modèle continué

On évalue les deux versions **sans recharger** : la base via `model.disable_adapter()`,
le modèle continué adaptateur actif. FLORES `devtest`, six directions, BLEU + chrF.

In [ ]:
from datasets import load_dataset
from sacrebleu import corpus_bleu, corpus_chrf
from contextlib import nullcontext

flores_split = "devtest"
flores_texts = {}
try:
    per_id = {}
    for short, code_ in FLORES_LANGS.items():
        ds = load_dataset("openlanguagedata/flores_plus", code_, split=flores_split)
        per_id[short] = {str(r["id"]): r["text"] for r in ds}
    common = sorted(set.intersection(*[set(d) for d in per_id.values()]))
    flores_texts = {short: [per_id[short][i] for i in common] for short in FLORES_LANGS}
except Exception as exc:
    print(f"FLORES+ indisponible ({type(exc).__name__}), repli facebook/flores 'all'.")
    ds = load_dataset("facebook/flores", "all", split=flores_split, trust_remote_code=True)
    flores_texts = {short: [r[f"sentence_{code_}"] for r in ds] for short, code_ in FLORES_LANGS.items()}

n_flores = len(next(iter(flores_texts.values())))
rows = []
for sgl in FLORES_LANGS:
    for tgl in FLORES_LANGS:
        if sgl == tgl:
            continue
        for i in range(n_flores):
            rows.append({"src_lang": sgl, "tgt_lang": tgl,
                         "src_text": flores_texts[sgl][i], "tgt_text": flores_texts[tgl][i]})
flores_df = pd.DataFrame(rows)
print("FLORES", flores_split, ":", n_flores, "phrases/langue,", len(flores_df), "paires")

model.eval()
cft_bf16 = (model.device.type == "cuda") and torch.cuda.is_bf16_supported()

def batched(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]

@torch.no_grad()
def generate(group, use_adapter):
    tokenizer.src_lang = FLORES_LANGS[group.iloc[0]["src_lang"]]
    bos = tokenizer.convert_tokens_to_ids(FLORES_LANGS[group.iloc[0]["tgt_lang"]])
    adapter_ctx = nullcontext() if use_adapter else model.disable_adapter()
    autocast_ctx = torch.autocast("cuda", dtype=torch.bfloat16) if cft_bf16 else nullcontext()
    preds = []
    with adapter_ctx:
        for batch in batched([str(x) for x in group["src_text"].tolist()], EVAL_BATCH_SIZE):
            enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_SOURCE_LENGTH).to(model.device)
            with autocast_ctx:
                gen = model.generate(**enc, forced_bos_token_id=bos,
                                     max_new_tokens=MAX_TARGET_LENGTH, num_beams=4)
            preds.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

reports = []
for (sl, tl), group in flores_df.groupby(["src_lang", "tgt_lang"], sort=True):
    group = group.sample(n=min(MAX_EVAL_PER_DIRECTION, len(group)), random_state=42).reset_index(drop=True)
    refs = group["tgt_text"].astype(str).tolist()
    base_p = generate(group, use_adapter=False)
    cft_p = generate(group, use_adapter=True)
    reports.append({
        "direction": f"{sl}->{tl}", "rows": len(group),
        "base_bleu": round(float(corpus_bleu(base_p, [refs]).score), 2),
        "cft_bleu": round(float(corpus_bleu(cft_p, [refs]).score), 2),
        "base_chrf": round(float(corpus_chrf(base_p, [refs]).score), 2),
        "cft_chrf": round(float(corpus_chrf(cft_p, [refs]).score), 2),
    })

flores_cft = pd.DataFrame(reports)
flores_cft["d_bleu"] = (flores_cft["cft_bleu"] - flores_cft["base_bleu"]).round(2)
flores_cft["d_chrf"] = (flores_cft["cft_chrf"] - flores_cft["base_chrf"]).round(2)
flores_cft.loc["MOY"] = ["MOYENNE", flores_cft["rows"].sum(),
                         *flores_cft[["base_bleu", "cft_bleu", "base_chrf", "cft_chrf", "d_bleu", "d_chrf"]].mean().round(2)]
flores_cft.to_csv(OUTPUT_DIR / "flores_base_vs_cft.csv", index=False)
display(flores_cft)

## 8. Étude comparative : propre vs back-translation vs continué (test interne)

On charge les rapports d'évaluation **internes** des modèles précédents et on les met en
regard du modèle continué, direction par direction. C'est l'**ablation** qui montre
l'apport de chaque régime d'entraînement.

In [ ]:
test_df = pd.read_csv(TEST_CSV)

def eval_internal(group):
    tokenizer.src_lang = group.iloc[0]["src_lang_nllb"]
    bos = tokenizer.convert_tokens_to_ids(group.iloc[0]["tgt_lang_nllb"])
    preds = []
    with torch.no_grad():
        for batch in batched(group["src_text"].tolist(), EVAL_BATCH_SIZE):
            enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_SOURCE_LENGTH).to(model.device)
            gen = model.generate(**enc, forced_bos_token_id=bos,
                                 max_new_tokens=MAX_TARGET_LENGTH, num_beams=4)
            preds.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

cft_rows = []
for (sl, tl), group in test_df.groupby(["src_lang", "tgt_lang"], sort=True):
    group = group.sample(n=min(MAX_EVAL_PER_DIRECTION, len(group)), random_state=42).reset_index(drop=True)
    refs = group["tgt_text"].astype(str).tolist()
    preds = eval_internal(group)
    cft_rows.append({"src_lang": sl, "tgt_lang": tl,
                     "bleu_cft": round(float(corpus_bleu(preds, [refs]).score), 2)})
cft_int = pd.DataFrame(cft_rows)
cft_int.to_csv(OUTPUT_DIR / "evaluation_report.csv", index=False)

cmp = cft_int.copy()
cmp["direction"] = cmp["src_lang"] + "->" + cmp["tgt_lang"]
for name, path in COMPARE_DIRS.items():
    p = Path(path)
    if p.exists():
        other = pd.read_csv(p)[["src_lang", "tgt_lang", "bleu"]].rename(columns={"bleu": name})
        cmp = cmp.merge(other, on=["src_lang", "tgt_lang"], how="left")
cols = ["direction"] + [c for c in COMPARE_DIRS if c in cmp.columns] + ["bleu_cft"]
cmp = cmp[cols].rename(columns={"bleu_cft": "Continué"})
display(cmp)
print("Adopter le modèle continué s'il améliore en<->ewe sans régresser sur les autres directions.")

## 9. Décision

- Si le modèle continué **gagne sur en↔ewe sans régresser ailleurs**, il devient le
  meilleur candidat : pointer `ADAPTER_PATH=models/nllb-ewe-lora-cft` dans le `.env`.
- Sinon, on conserve le modèle back-translation. Dans tous les cas, l'expérience est
  **documentée et comparée** — c'est la démarche attendue en recherche low-resource MT.

**Limites & honnêteté** : un gain sur le test interne doit être confirmé sur FLORES ; la
qualité des cibles éwé reste le facteur limitant. Une itération de back-translation (round 2)
et davantage d'époques constituent les prolongements naturels.